# Desafio DIO: Assistente de Investimentos com RPA e IA Generativa

Este notebook faz a extração automatizada de clientes de uma página web simulada (RPA) e envia os dados estruturados para um Webhook no **N8N**.
O N8N cruza o perfil de cada cliente com a base de investimentos (`data.csv`) e utiliza o **Google Gemini** para gerar recomendações personalizadas.

## 1. Instalação das Dependências

In [ ]:
!pip install requests beautifulsoup4

## 2. Extração via RPA (Python + BeautifulSoup)

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"

# 1) Baixar a página
response = requests.get(URL_PAGINA)
response.raise_for_status()
html = response.text

# 2) Fazer o scraping da tabela de clientes
soup = BeautifulSoup(html, "html.parser")
linhas = soup.select("#clientes tbody tr")

clientes = []
for linha in linhas:
    colunas = linha.find_all("td")
    if len(colunas) >= 4:
        cliente = {
            "nome": colunas[0].get_text(strip=True),
            "email": colunas[1].get_text(strip=True),
            "saldo": colunas[2].get_text(strip=True),
            "perfil": colunas[3].get_text(strip=True),
        }
        clientes.append(cliente)

print(f"Foram encontrados {len(clientes)} clientes:\n")
for c in clientes:
    print(f"• {c['nome']} | {c['perfil']} | {c['saldo']} | {c['email']}")


## 3. Envio para o Webhook do N8N e Processamento com IA

In [ ]:
# URL do Webhook do N8N (produção)
N8N_WEBHOOK = "https://testtest222.app.n8n.cloud/webhook/clientes"

print(f"Enviando dados para o N8N: {N8N_WEBHOOK}\n")
response_webhook = requests.post(N8N_WEBHOOK, json={"clientes": clientes})

print(f"Status HTTP: {response_webhook.status_code}\n")
resultado = response_webhook.json()

print("=" * 60)
print("RESULTADO DO PROCESSAMENTO NO N8N + GEMINI")
print("=" * 60)
for item in resultado.get("clientes", []):
    print(f"\n👤 Cliente: {item['cliente']} ({item['perfil']} - Saldo: {item['saldo']})")
    print("🤖 Mensagem Gerada pela IA:")
    print("-" * 50)
    print(item.get("recomendacao_ia", "").strip())
    print("-" * 50)
